# ☀️ Aditya-L1 Solar Flare Forecasting — 4-Horizon GPU Benchmark
**ISRO Aditya-L1 SoLEXS & HEL1OS Multi-Instrument Solar Physics & Deep Learning Pipeline**

This notebook evaluates **10 Machine Learning & Deep Learning Architectures** across **4 Forecasting Horizons** ($F_1, F_3, F_6, F_{12}$) under zero-leakage `StratifiedGroupKFold` cross-validation grouped by observation date.


In [ ]:
# Cell 1: System Inspection & GPU Environment Setup
import os, sys, glob, zipfile, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')


In [ ]:
# Cell 2: Locate and Extract dataset_forecast_v1 Files
print('🔍 Locating forecasting dataset files in /kaggle/input...')
npy_files = glob.glob('/kaggle/input/**/*.npy', recursive=True) + glob.glob('**/*.npy', recursive=True)
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True) + glob.glob('**/*.csv', recursive=True)

X_file = next((f for f in npy_files if 'X_forecast' in f or 'X_sequences' in f), None)
y_file = next((f for f in npy_files if 'y_forecast' in f or 'y_labels' in f), None)
meta_file = next((f for f in csv_files if 'forecast_metadata' in f or 'sequence_metadata' in f), None)

models_py = glob.glob('/kaggle/input/**/models.py', recursive=True) + glob.glob('**/models.py', recursive=True)
if models_py:
    m_dir = os.path.dirname(models_py[0])
    if m_dir not in sys.path:
        sys.path.insert(0, m_dir)

X = np.load(X_file).astype(np.float32)
y = np.load(y_file).astype(np.int64)
df_meta = pd.read_csv(meta_file)

print(f'Loaded X Shape: {X.shape}, y Shape: {y.shape}')
print(f'Unique Observation Dates: {df_meta["observation_date"].nunique()}')


In [ ]:
# Cell 3: 4-Horizon Sub-Dataset Partitioning (F1, F3, F6, F12)
horizons = {'Dataset_F1': 60.0, 'Dataset_F3': 180.0, 'Dataset_F6': 360.0, 'Dataset_F12': 720.0}
ds_summary = []

for ds_name, max_m in horizons.items():
    mask = (y == 0) | ((y == 1) & (df_meta['minutes_until_flare'] <= max_m))
    X_sub, y_sub = X[mask], y[mask]
    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))
    ds_summary.append({
        'Horizon_Dataset': ds_name,
        'Max_Horizon': f'{max_m/60:.0f}h',
        'Total_Samples': len(y_sub),
        'Positives': n_pos,
        'Negatives': n_neg,
        'Class_Ratio': f'{n_neg/n_pos:.2f}:1'
    })

df_horizons = pd.DataFrame(ds_summary)
print('=== FORECASTING HORIZON SUB-DATASETS ===')
display(df_horizons)


In [ ]:
# Cell 4: Run Full 10-Model Benchmarking Pipeline
from models import MODEL_REGISTRY
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

groups = df_meta['observation_date'].values
sgkf = StratifiedGroupKFold(n_splits=5)
splits = list(sgkf.split(X, y, groups=groups))

# Feature summary for tabular baselines
X_flat = X.reshape(len(y), -1)
X_feats = np.column_stack([
    np.max(X_flat, axis=1), np.mean(X_flat, axis=1),
    np.std(X_flat, axis=1), np.percentile(X_flat, 95, axis=1)
])

results = []

# 1. LR
lr_m = []
for tr, va in splits:
    m, s = np.mean(X_feats[tr], axis=0), np.std(X_feats[tr], axis=0) + 1e-8
    clf = LogisticRegression().fit((X_feats[tr] - m)/s, y[tr])
    p = clf.predict_proba((X_feats[va] - m)/s)[:, 1]
    pred = (p > 0.5).astype(int)
    lr_m.append({'acc': accuracy_score(y[va], pred), 'prec': precision_score(y[va], pred, zero_division=0),
                 'rec': recall_score(y[va], pred, zero_division=0), 'f1': f1_score(y[va], pred, zero_division=0),
                 'roc': roc_auc_score(y[va], p), 'pr': average_precision_score(y[va], p)})
results.append({'model': 'Logistic Regression', 'f1': np.mean([m['f1'] for m in lr_m]), 'roc_auc': np.mean([m['roc'] for m in lr_m])})

# 2. RF
rf_m = []
for tr, va in splits:
    clf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_feats[tr], y[tr])
    p = clf.predict_proba(X_feats[va])[:, 1]
    pred = (p > 0.5).astype(int)
    rf_m.append({'acc': accuracy_score(y[va], pred), 'prec': precision_score(y[va], pred, zero_division=0),
                 'rec': recall_score(y[va], pred, zero_division=0), 'f1': f1_score(y[va], pred, zero_division=0),
                 'roc': roc_auc_score(y[va], p), 'pr': average_precision_score(y[va], p)})
results.append({'model': 'Random Forest', 'f1': np.mean([m['f1'] for m in rf_m]), 'roc_auc': np.mean([m['roc'] for m in rf_m])})

# 3. XGBoost
xgb_m = []
for tr, va in splits:
    clf = xgb.XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss').fit(X_feats[tr], y[tr])
    p = clf.predict_proba(X_feats[va])[:, 1]
    pred = (p > 0.5).astype(int)
    xgb_m.append({'acc': accuracy_score(y[va], pred), 'prec': precision_score(y[va], pred, zero_division=0),
                  'rec': recall_score(y[va], pred, zero_division=0), 'f1': f1_score(y[va], pred, zero_division=0),
                  'roc': roc_auc_score(y[va], p), 'pr': average_precision_score(y[va], p)})
results.append({'model': 'XGBoost', 'f1': np.mean([m['f1'] for m in xgb_m]), 'roc_auc': np.mean([m['roc'] for m in xgb_m])})

# 4-10. DL Models
for name, cls in MODEL_REGISTRY.items():
    dl_m = []
    for tr, va in splits:
        m_v, s_v = np.mean(X[tr], axis=(0,1), keepdims=True), np.std(X[tr], axis=(0,1), keepdims=True) + 1e-8
        ds_tr = TensorDataset(torch.from_numpy((X[tr]-m_v)/s_v), torch.from_numpy(y[tr]))
        ds_va = TensorDataset(torch.from_numpy((X[va]-m_v)/s_v), torch.from_numpy(y[va]))
        ldr_tr, ldr_va = DataLoader(ds_tr, batch_size=32, shuffle=True), DataLoader(ds_va, batch_size=32, shuffle=False)
        model = cls(in_channels=4, num_classes=2).to(device)
        opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        for epoch in range(1, 10):
            model.train()
            for xb, yb in ldr_tr:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = crit(model(xb), yb)
                loss.backward()
                opt.step()
        model.eval()
        p_list, t_list = [], []
        with torch.no_grad():
            for xb, yb in ldr_va:
                p = torch.softmax(model(xb.to(device)), dim=1)[:, 1]
                p_list.extend(p.cpu().numpy())
                t_list.extend(yb.numpy())
        y_t, y_p = np.array(t_list), np.array(p_list)
        dl_m.append({'f1': f1_score(y_t, (y_p>0.5).astype(int), zero_division=0), 'roc': roc_auc_score(y_t, y_p)})
    results.append({'model': name.upper(), 'f1': np.mean([m['f1'] for m in dl_m]), 'roc_auc': np.mean([m['roc'] for m in dl_m])})

df_res = pd.DataFrame(results).sort_values(by='f1', ascending=False)
print('=== FINAL 10-MODEL BENCHMARK RESULTS ===')
display(df_res)
